# Fraud Detection Model Testing

This notebook tests the accuracy, fairness, and robustness of the trained fraud detection models.

In [1]:
import pandas as pd
import numpy as np
import onnxruntime as rt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

## 1. Data Loading and Preprocessing
We load the dataset and apply the same preprocessing steps as used during training.

In [2]:
# Load the dataset
data_path = '../data/investigation_train_large_checked.csv'
try:
    df = pd.read_csv(data_path)
    print(f"Loaded dataset with shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at {data_path}")

# Preprocessing
# Drop 'Ja' and 'Nee' columns if they exist
cols_to_drop = [col for col in ['Ja', 'Nee'] if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")

# Convert 'checked' target column to integer (0/1)
if df['checked'].dtype == 'bool' or df['checked'].dtype == 'object':
    df['checked'] = df['checked'].astype(int)

print("Target distribution:")
print(df['checked'].value_counts())

# Split into X and y
X = df.drop(columns=['checked'])
y = df['checked']

# Create a test set
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Test set shape: {X_test.shape}")

Loaded dataset with shape: (130000, 318)
Dropped columns: ['Ja', 'Nee']
Target distribution:
checked
0    110496
1     19504
Name: count, dtype: int64
Test set shape: (26000, 315)


## 2. Model Loading and Inference Helpers
Functions to load ONNX models and generate predictions.

In [3]:
def load_model(model_path):
    """Loads an ONNX model session."""
    try:
        session = rt.InferenceSession(model_path)
        print(f"Successfully loaded model from {model_path}")
        return session
    except Exception as e:
        print(f"Error loading model {model_path}: {e}")
        return None

def predict_onnx(session, X_input):
    """
    Generates predictions using an ONNX session.
    Assumes all required fields exist in X_input.
    """
    input_meta = session.get_inputs()
    inputs = {}
    
    for meta in input_meta:
        name = meta.name
        # If the model expects specific features by name, we extract them.
        if name in X_input.columns:
            inputs[name] = X_input[name].values.astype(np.float32).reshape(-1, 1)
        elif len(input_meta) == 1:
            # Fallback for single matrix input
            inputs[name] = X_input.to_numpy(dtype=np.float32)

    output_names = [meta.name for meta in session.get_outputs()]
    try:
        preds = session.run(output_names, inputs)
        return preds[0]
    except Exception as e:
        print(f"Prediction error: {e}")
        return None

## 3. Standard Evaluation
We define metrics and the specific feature sets for each model.
Model 1 (Good) excludes proxy variables.
Model 2 (Bad) includes them.

In [4]:
# Partitioning + metamorphic configs 

def _partition_by_contacts(df):
    contact_cols = [c for c in df.columns if c.startswith("contacten_")]
    if not contact_cols:
        return None, None
    contact_totals = df[contact_cols].sum(axis=1)
    top_cut = contact_totals.quantile(0.8)
    bottom_cut = contact_totals.quantile(0.2)
    return df[contact_totals >= top_cut], df[contact_totals <= bottom_cut]


def _partition_by_reintegration(df):
    reintegr_cols = [c for c in df.columns if c.startswith("deelname_act_reintegratieladder")]
    if not reintegr_cols:
        return None, None
    activity_sum = df[reintegr_cols].sum(axis=1)
    return df[activity_sum > 0], df[activity_sum == 0]


partition_tests = [
    {
        "name": "District: Charlois vs Kralingen_C",
        "required_cols": ["adres_recentste_wijk_charlois", "adres_recentste_wijk_kralingen_c"],
        "labels": ("Charlois = 1", "Kralingen_C = 1"),
        "group_fn": lambda df: (
            df[df["adres_recentste_wijk_charlois"] == 1],
            df[df["adres_recentste_wijk_kralingen_c"] == 1],
        ),
    },
    {
        "name": "Mobility: wijk count <= 1 vs >= 4",
        "required_cols": ["adres_aantal_verschillende_wijken"],
        "labels": ("Wijk count <= 1", "Wijk count >= 4"),
        "group_fn": lambda df: (
            df[df["adres_aantal_verschillende_wijken"] <= 1],
            df[df["adres_aantal_verschillende_wijken"] >= 4],
        ),
    },
    {
        "name": "Children: has children vs none",
        "required_cols": ["relatie_kind_huidige_aantal"],
        "labels": ("relatie_kind_huidige_aantal > 0", "relatie_kind_huidige_aantal = 0"),
        "group_fn": lambda df: (
            df[df["relatie_kind_huidige_aantal"] > 0],
            df[df["relatie_kind_huidige_aantal"] == 0],
        ),
    },
    {
        "name": "Engagement: contacten_* top vs bottom 20%",
        "required_cols": [],
        "labels": ("Top 20% contacten_*", "Bottom 20% contacten_*"),
        "group_fn": _partition_by_contacts,
    },
    {
        "name": "Reintegration: active vs inactive",
        "required_cols": [],
        "labels": ("deelname_act_reintegratieladder_* > 0", "deelname_act_reintegratieladder_* = 0"),
        "group_fn": _partition_by_reintegration,
    },
    {
        "name": "Age: <=30 vs >=55",
        "required_cols": ["persoon_leeftijd_bij_onderzoek"],
        "labels": ("Age <= 30", "Age >= 55"),
        "group_fn": lambda df: (
            df[df["persoon_leeftijd_bij_onderzoek"] <= 30],
            df[df["persoon_leeftijd_bij_onderzoek"] >= 55],
        ),
    },
]


metamorphic_relations = [
    {
        "name": "adres_aantal_verschillende_wijken - 1",
        "required_cols": ["adres_aantal_verschillende_wijken"],
        "transform": lambda df: df.assign(
            adres_aantal_verschillende_wijken=np.maximum(df["adres_aantal_verschillende_wijken"] - 1, 0)
        ),
        "expectation": "Small mobility change should not flip decision.",
    },
    {
        "name": "Move Charlois -> Kralingen_C",
        "required_cols": ["adres_recentste_wijk_charlois", "adres_recentste_wijk_kralingen_c"],
        "transform": lambda df: df.assign(
            adres_recentste_wijk_charlois=0,
            adres_recentste_wijk_kralingen_c=1,
        ),
        "expectation": "Changing neighbourhood alone should not alter outcome.",
    },
    {
        "name": "Spreektaal: Nederlands -> Anders",
        "required_cols": ["persoonlijke_eigenschappen_spreektaal"],
        "transform": lambda df: df.assign(
            persoonlijke_eigenschappen_spreektaal=0,
            persoonlijke_eigenschappen_spreektaal_anders=1
            if "persoonlijke_eigenschappen_spreektaal_anders" in df.columns
            else df.get("persoonlijke_eigenschappen_spreektaal_anders", 1),
        ),
        "expectation": "Language switch alone should not change prediction.",
    },
    {
        "name": "Flip persoon_geslacht_vrouw",
        "required_cols": ["persoon_geslacht_vrouw"],
        "transform": lambda df: df.assign(persoon_geslacht_vrouw=1 - df["persoon_geslacht_vrouw"]),
        "expectation": "Gender swap should not affect outcome.",
    },
    {
        "name": "Set belemmering_psychische_problemen = 1",
        "required_cols": ["belemmering_psychische_problemen"],
        "transform": lambda df: df.assign(belemmering_psychische_problemen=1),
        "expectation": "Marking psychological problems should not change decision.",
    },
    {
        "name": "relatie_kind_huidige_aantal + 1",
        "required_cols": ["relatie_kind_huidige_aantal"],
        "transform": lambda df: df.assign(relatie_kind_huidige_aantal=df["relatie_kind_huidige_aantal"] + 1),
        "expectation": "One extra child should not swing prediction.",
    },
]


def evaluate_model(session, X_test_all, y_test):
    """
    Evaluates the model on the test set.
    """
    print(f"--- Evaluating Model ---")
    
    # Predict (predict_onnx handles feature selection)
    y_pred = predict_onnx(session, X_test_all)
    
    if y_pred is None:
        print("Skipping evaluation due to prediction failure.")
        return
        
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return y_pred

## 4. Fairness Testing (Partitioning)
We partition the test data based on sensitive attributes (e.g., language barrier) and compare the False Positive Rate (FPR) between groups.
- **Hypothesis**: A fair model should have similar FPR across groups. A biased model might flag one group more often incorrectly.

In [5]:
def calculate_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if (fp + tn) == 0:
        return 0.0
    return fp / (fp + tn)

def test_partitioning(session, X_test_all, y_test):
    """
    Tests fairness by partitioning data on proxy variables.
    """
    print(f"\n--- Fairness Test (Partitioning) ---")
    
    # Predict (predict_onnx handles feature selection)
    y_pred_all = predict_onnx(session, X_test_all)
    if y_pred_all is None:
        return

    # Create a dataframe for easy slicing
    results_df = X_test_all.copy()
    results_df['target'] = y_test.values
    results_df['prediction'] = y_pred_all
    
    for spec in partition_tests:
        missing = [c for c in spec.get("required_cols", []) if c not in results_df.columns]
        if missing:
            print(f"\nSkipping {spec['name']}: missing columns {missing}")
            continue

        try:
            group_1, group_0 = spec["group_fn"](results_df)
        except Exception as e:
            print(f"\nSkipping {spec['name']}: error while partitioning ({e})")
            continue

        if group_1 is None or group_0 is None:
            print(f"\nSkipping {spec['name']}: unable to build partitions.")
            continue

        print(f"\nPartitioning by: {spec['name']}")
        label_1, label_0 = spec.get("labels", ("Group 1", "Group 0"))

        if len(group_1) == 0 or len(group_0) == 0:
            print("  Skipping: One partition is empty.")
            continue

        fpr_1 = calculate_fpr(group_1["target"], group_1["prediction"])
        fpr_0 = calculate_fpr(group_0["target"], group_0["prediction"])

        count_1 = len(group_1)
        count_0 = len(group_0)

        print(f"  {label_1} (n={count_1}): FPR = {fpr_1:.4f}")
        print(f"  {label_0} (n={count_0}): FPR = {fpr_0:.4f}")
        print(f"  Diff (G1 - G0): {fpr_1 - fpr_0:.4f}")
        
        if abs(fpr_1 - fpr_0) > 0.05:
            print("\033[91m  (!) Significant disparity detected (> 5%)\033[0m")
        else:
            print("\033[92m  (OK) Disparity within 5%\033[0m")

## 5. Metamorphic Testing
We apply a metamorphic transformation by flipping sensitive attributes. The prediction should remain unchanged if the model is fair and does not use the attribute.

In [6]:
def test_metamorphic(session, X_test_all, sample_size=1000):
    print(f"\n--- Metamorphic Test ---")
    
    # Select a sample to speed up testing
    sample = X_test_all.sample(n=min(sample_size, len(X_test_all)), random_state=42).copy()
    
    # Original prediction
    preds_orig = predict_onnx(session, sample)
    if preds_orig is None:
        print("Original predictions failed.")
        return

    robustness_scores = {}

    for relation in metamorphic_relations:
        missing = [c for c in relation.get("required_cols", []) if c not in sample.columns]
        if missing:
            print(f"Skipping {relation['name']}: missing columns {missing}")
            continue

        transformed = relation["transform"](sample.copy())
        preds_new = predict_onnx(session, transformed)
        if preds_new is None:
            print(f"Skipping {relation['name']}: prediction failed after transform")
            continue
            
        changes = np.sum(preds_orig != preds_new)
        total = len(sample)
        consistency = (total - changes) / total
        
        robustness_scores[relation["name"]] = consistency
        expectation = relation.get("expectation", "Should remain stable.")
        print(f"Relation: {relation['name']:40} | Score: {consistency:.4f} | Expectation: {expectation}")
        
        if consistency < 1.0:
            print(f"\033[91m    -> FAIL: {changes} changes detected.\033[0m")
    
    # Summary
    print("\nRobustness Summary:")
    pass_count = sum(1 for s in robustness_scores.values() if s == 1.0)
    print(f"Passed: {pass_count}/{len(robustness_scores)} relations.")
    if pass_count < len(robustness_scores):
        print("\033[91mSome metamorphic relations affect the model predictions!\033[0m")
    else:
        print("\033[92mModel is robust to all metamorphic relations tested.\033[0m")

## 6. Execution
Run the full test suite for the selected model.

In [15]:
# Model Selection
# Change this path to test a different model
MODEL_PATH = "model_2.onnx"
# MODEL_PATH = "model_2.onnx"

print(f"Loading model from: {MODEL_PATH}")
session = load_model(MODEL_PATH)

if session:
    evaluate_model(session, X_test, y_test)
    test_partitioning(session, X_test, y_test)
    test_metamorphic(session, X_test)
else:
    print("Model loading failed.")

Loading model from: model_2.onnx
Successfully loaded model from model_2.onnx
--- Evaluating Model ---
Accuracy:  0.9577
Precision: 0.8770
Recall:    0.8354
F1 Score:  0.8557

Confusion Matrix:
[[21642   457]
 [  642  3259]]

--- Fairness Test (Partitioning) ---

Partitioning by: District: Charlois vs Kralingen_C
  Charlois = 1 (n=2809): FPR = 0.0219
  Kralingen_C = 1 (n=1146): FPR = 0.0117
  Diff (G1 - G0): 0.0102
  (OK) Disparity within 5%

Partitioning by: Mobility: wijk count <= 1 vs >= 4
  Wijk count <= 1 (n=7197): FPR = 0.0179
  Wijk count >= 4 (n=1919): FPR = 0.0275
  Diff (G1 - G0): -0.0097
  (OK) Disparity within 5%

Partitioning by: Children: has children vs none
  relatie_kind_huidige_aantal > 0 (n=17093): FPR = 0.0241
  relatie_kind_huidige_aantal = 0 (n=8907): FPR = 0.0147
  Diff (G1 - G0): 0.0094
  (OK) Disparity within 5%

Partitioning by: Engagement: contacten_* top vs bottom 20%
  Top 20% contacten_* (n=5327): FPR = 0.0219
  Bottom 20% contacten_* (n=5336): FPR = 0.0204